# 10 — Pega-native NaiveBayes Feature Importance (L5B15)

Re-implements Pega's *"ADM NB Explained"* methodology directly on our ADM datamart exports — using only the **predictor-binning snapshots** (per-bin `pyBinPositives` / `pyBinNegatives` counts), no surrogate model required.

## Why this notebook exists alongside 06 / 09

| Notebook | Model used to compute importance | Source of truth |
|---|---|---|
| 06 (SHAP/LIME/DT) | CatBoost surrogate fit to Pega's propensity | Surrogate explanations |
| 09 (vendor XGBoost) | XGBoost on top-decile-binarised propensity | Surrogate explanations |
| **10 (this notebook)** | **None — direct from Pega's NB binning** | **Pega's own algorithm** |

This is what Pega itself uses for its in-platform feature-importance display, so the rankings here are the *reference* against which the thesis surrogates can be benchmarked.

## Pega's formula (`cdh_utils.feature_importance`)

For each predictor with bins `i = 1..n`:

$$
\text{LogOdds}_i = \log(\text{Pos}_i + 1/n) - \log(\sum \text{Pos} + 1) - \bigl[ \log(\text{Neg}_i + 1/n) - \log(\sum \text{Neg} + 1) \bigr]
$$

$$
\text{Importance} = \frac{\sum_i |\text{LogOdds}_i| \cdot \text{Responses}_i}{\sum_i \text{Responses}_i}
$$

$$
\text{ScaledImportance} = \frac{\text{Importance}}{\max(\text{Importance})} \times 100
$$

The `1/n` and `+1` are Laplace smoothing to avoid `log 0`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

from my_project.metrics import stability_spearman, jaccard_at_k

REPO = Path('..').resolve()
ADM_DIR = REPO / 'data' / 'adm'
ARTIFACTS = REPO / 'data' / 'artifacts' / 'l5b15'
OUT_DIR = ARTIFACTS / 'pega_nb'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The L5B15 Email/Outbound Luggage Click-Through model (24 active predictors).
L5B15_MODEL_ID = '340a718b-9899-5637-b3be-1b1e660ef365'


## 1. Load ADM snapshots and pick the L5B15 model

In [ ]:
models  = pl.read_ndjson(ADM_DIR / 'data-model-snapshop.json',           infer_schema_length=10000)
binning = pl.read_ndjson(ADM_DIR / 'data-Predictor-binning-snapshot.json', infer_schema_length=10000)

model_meta = (
    models.filter(pl.col('pyModelID') == L5B15_MODEL_ID)
    .sort('pySnapshotTime', descending=True)
    .head(1)
)
print('Selected L5B15 model:')
print(model_meta.select([
    'pyName','pyTreatment','pyGroup','pyChannel','pyDirection',
    'pyConfigurationName','pyPerformance','pyPositives','pyNegatives',
    'pyActivePredictors','pySnapshotTime']).to_pandas().T)

bins_l5 = binning.filter(pl.col('pyModelID') == L5B15_MODEL_ID)
print(f'\nBinning rows for this model: {bins_l5.shape[0]}')
print('EntryType counts:', dict(zip(
    bins_l5['pyEntryType'].value_counts()['pyEntryType'].to_list(),
    bins_l5['pyEntryType'].value_counts()['count'].to_list())))


## 2. Compute Pega-native NB importance per active predictor

We restrict to `pyEntryType == 'Active'` (24 predictors) — Pega's own importance only ranks active predictors because inactive ones don't contribute to the score.

In [ ]:
def nb_feature_importance(df_bins: pl.DataFrame) -> pl.DataFrame:
    """Pega's cdh_utils.feature_importance formula, in pure Polars.

    Returns one row per predictor with: Importance, ScaledImportance.
    Expects pyBinPositives / pyBinNegatives / pyBinResponseCount / pyTotalBins.
    """
    per_bin = df_bins.with_columns(
        n=pl.col('pyTotalBins').cast(pl.Int64),
        bin_pos=pl.col('pyBinPositives').cast(pl.Float64),
        bin_neg=pl.col('pyBinNegatives').cast(pl.Float64),
        bin_resp=pl.col('pyBinResponseCount').cast(pl.Float64),
    )
    per_pred = per_bin.group_by('pyPredictorName').agg(
        sum_pos=pl.col('bin_pos').sum(),
        sum_neg=pl.col('bin_neg').sum(),
        sum_resp=pl.col('bin_resp').sum(),
        n=pl.col('n').first(),
        bins_pos=pl.col('bin_pos'),
        bins_neg=pl.col('bin_neg'),
        bins_resp=pl.col('bin_resp'),
    )

    records = []
    for r in per_pred.iter_rows(named=True):
        n = r['n'] if r['n'] and r['n'] > 0 else max(len(r['bins_pos']), 1)
        bp = np.asarray(r['bins_pos'], dtype=float)
        bn = np.asarray(r['bins_neg'], dtype=float)
        br = np.asarray(r['bins_resp'], dtype=float)
        sp, sn = bp.sum(), bn.sum()
        if sp == 0 or sn == 0 or br.sum() == 0:
            imp = 0.0
        else:
            log_odds = (
                np.log(bp + 1.0 / n) - np.log(sp + 1.0)
                - (np.log(bn + 1.0 / n) - np.log(sn + 1.0))
            )
            imp = float(np.sum(np.abs(log_odds) * br) / br.sum())
        records.append({'pyPredictorName': r['pyPredictorName'], 'Importance': imp})

    out = pl.DataFrame(records).sort('Importance', descending=True)
    max_imp = out['Importance'].max() or 1.0
    return out.with_columns(
        ScaledImportance=pl.col('Importance') / max_imp * 100
    )


active_bins = bins_l5.filter(pl.col('pyEntryType') == 'Active')
importance = nb_feature_importance(active_bins)
importance.to_pandas()


In [ ]:
def category_of(field: str) -> str:
    if '.' in field:
        return field.split('.', 1)[0]
    return 'Internal'

CATEGORY_COLORS = {
    'CurrentContext': '#FFA500', 'Context': '#FFA500',
    'IH':             '#2CA02C',
    'Customer':       '#1F77B4',
    'Param':          '#17BECF', 'param::Param': '#17BECF',
    'CustBookedFlight': '#9467BD',
    'PreComputes':    '#008080',
    'Internal':       '#7F7F7F',
}

imp_pd = importance.to_pandas()
imp_pd['Category'] = imp_pd['pyPredictorName'].map(category_of)
imp_pd['Color']    = imp_pd['Category'].map(lambda c: CATEGORY_COLORS.get(c, '#888'))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=imp_pd['ScaledImportance'][::-1],
    y=imp_pd['pyPredictorName'][::-1],
    orientation='h',
    marker_color=imp_pd['Color'][::-1],
    hovertext=[f'{r.Importance:.4f}' for r in imp_pd.itertuples()][::-1],
    hoverinfo='text+x',
    showlegend=False,
))
for cat, color in CATEGORY_COLORS.items():
    if cat in imp_pd['Category'].values:
        fig.add_trace(go.Bar(x=[None], y=[None], name=cat, marker_color=color, showlegend=True, hoverinfo='skip'))
fig.update_layout(
    title='Pega-native NB feature importance — L5B15 (scaled, 100 = max)',
    xaxis_title='Scaled importance (0–100)',
    yaxis=dict(automargin=True),
    height=700, width=1100, template='plotly_white',
    legend=dict(title='Category', orientation='h', y=-0.10),
    barmode='group',
)
fig.show()

(importance.with_columns(pyModelID=pl.lit(L5B15_MODEL_ID))
    .write_csv(OUT_DIR / 'pega_nb_importance.csv'))
print(f'Wrote {OUT_DIR / "pega_nb_importance.csv"}')


## 3. Worked example for the top predictor

For sanity-checking against Pega's notebook: reproduce per-bin Lift, Z-Ratio and log-odds for the top-importance predictor. These are exactly the columns shown in Pega's binning report.

In [ ]:
top_pred = importance['pyPredictorName'][0]
print(f'Top predictor: {top_pred}')

bins_top = active_bins.filter(pl.col('pyPredictorName') == top_pred).sort('pyBinIndex')
n = int(bins_top['pyTotalBins'][0])
bp = bins_top['pyBinPositives'].cast(pl.Float64).to_numpy()
bn = bins_top['pyBinNegatives'].cast(pl.Float64).to_numpy()
br = bins_top['pyBinResponseCount'].cast(pl.Float64).to_numpy()
sp, sn = bp.sum(), bn.sum()

pos_frac = bp / sp
neg_frac = bn / sn
propensity = bp / np.maximum(bp + bn, 1)
avg_propensity = sp / (sp + sn)
lift = propensity / avg_propensity
z_denom = np.sqrt(pos_frac * (1 - pos_frac) / sp + neg_frac * (1 - neg_frac) / sn)
z_ratio = (pos_frac - neg_frac) / np.where(z_denom == 0, np.nan, z_denom)
log_odds = (np.log(bp + 1.0 / n) - np.log(sp + 1.0)
            - (np.log(bn + 1.0 / n) - np.log(sn + 1.0)))

detail = pd.DataFrame({
    'Bin':         bins_top['pyBinSymbol'].to_list(),
    'Positives':   bp,
    'Negatives':   bn,
    'Responses%':  br / br.sum() * 100,
    'Pos %':       pos_frac * 100,
    'Neg %':       neg_frac * 100,
    'Propensity%': propensity * 100,
    'Lift':        lift,
    'ZRatio':      z_ratio,
    'LogOdds':     log_odds,
})
detail.round(4)


## 4. Compare to thesis SHAP / LIME / DT and to vendor XGBoost

The thesis methods (06) and the vendor XGBoost (09) both store their importances keyed by the dot-notation predictor names — matching Pega's `pyPredictorName` directly.

In [ ]:
import joblib

shap_imp = json.load(open(ARTIFACTS / 'shap_importances.json'))
lime_imp = json.load(open(ARTIFACTS / 'lime_importances.json'))

feature_cols_thesis = json.load(open(ARTIFACTS / 'feature_cols.json'))
dt_obj = joblib.load(ARTIFACTS / 'dt_model.pkl')
if isinstance(dt_obj, dict):
    dt_tree = dt_obj.get('tree') or dt_obj.get('model')
elif isinstance(dt_obj, tuple):
    dt_tree = dt_obj[0]
else:
    dt_tree = dt_obj
dt_imp = dict(zip(feature_cols_thesis, dt_tree.feature_importances_.tolist()))

# Vendor XGBoost importances. They live under hds-style names (Customer_X etc.),
# so map them back to dot-notation. We only need the active predictors anyway.
xgb_path = ARTIFACTS / 'vendor_hds' / 'vendor_xgb_importances.json'
xgb_raw = json.load(open(xgb_path)) if xgb_path.exists() else {}

def hds_to_dot(name: str) -> str:
    """Best-effort inverse of notebook 09's rename_to_hds for the dotted PredictorName space."""
    if name.startswith('Param_'):
        return 'param::Param.' + name[len('Param_'):].replace('_', '.')
    if name.startswith('Booking_'):
        return 'CustBookedFlight.' + name[len('Booking_'):].replace('_', '.')
    if name.startswith('Context_'):
        return 'CurrentContext.' + name[len('Context_'):].replace('_', '.')
    if name.startswith('Precompute_'):
        return 'PreComputes.' + name[len('Precompute_'):].replace('_', '.')
    if name.startswith('IH_'):
        return 'IH.' + name[len('IH_'):].replace('_', '.')
    if name.startswith('Customer_'):
        return 'Customer.' + name[len('Customer_'):].replace('_', '.')
    return name

xgb_imp = {hds_to_dot(k): v for k, v in xgb_raw.items()}

nb_imp = dict(zip(importance['pyPredictorName'].to_list(),
                  importance['Importance'].to_list()))
print(f'NB active predictors: {len(nb_imp)} | SHAP: {len(shap_imp)} | LIME: {len(lime_imp)} | DT: {len(dt_imp)} | XGB-mapped: {len(xgb_imp)}')


In [ ]:
shared = sorted(set(nb_imp) & set(shap_imp) & set(lime_imp) & set(dt_imp) & set(xgb_imp))
print(f'Shared predictors across all 5 methods: {len(shared)}')
missing_in_xgb = sorted(set(nb_imp) - set(xgb_imp))
if missing_in_xgb:
    print(f'NB-active features missing from XGB ranking: {missing_in_xgb}')

def rank_series(d, keys):
    s = pd.Series({k: d.get(k, 0.0) for k in keys})
    return s.rank(ascending=False, method='min')

methods = {
    'Pega NB': nb_imp, 'SHAP': shap_imp, 'LIME': lime_imp,
    'DT': dt_imp, 'XGB-vendor': xgb_imp,
}
ranks = pd.DataFrame({m: rank_series(d, shared) for m, d in methods.items()})

summary_rows = []
names = list(methods.keys())
for i, a in enumerate(names):
    for b in names[i+1:]:
        rho = stability_spearman(ranks[a], ranks[b])
        j5 = jaccard_at_k(ranks[a], ranks[b], 5)
        j10 = jaccard_at_k(ranks[a], ranks[b], 10)
        summary_rows.append({'A': a, 'B': b,
                             'Spearman ρ': round(rho, 4),
                             'Jaccard@5':  round(j5, 4),
                             'Jaccard@10': round(j10, 4)})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / 'nb_vs_others_comparison.csv', index=False)
summary


In [ ]:
from plotly.subplots import make_subplots

def short_name(f: str) -> str:
    parts = f.replace('::', '.').split('.')
    if len(parts) > 3:
        return '…' + '.'.join(parts[-3:])
    return '.'.join(parts[1:]) if len(parts) > 1 else f

def top_k_records(d, k, method):
    s = pd.Series(d).sort_values(ascending=False).head(k)
    return pd.DataFrame({
        'Feature':    s.index,
        'Short':      [short_name(f) for f in s.index],
        'Category':   [category_of(f) for f in s.index],
        'Importance': s.values,
        'Method':     method,
    })

panels = [top_k_records(d, 15, m) for m, d in methods.items()]
fig = make_subplots(
    rows=3, cols=2,
    horizontal_spacing=0.32, vertical_spacing=0.08,
    subplot_titles=[p['Method'].iloc[0] for p in panels] + [''],
)
positions = [(1,1),(1,2),(2,1),(2,2),(3,1)]
for panel, (r, c) in zip(panels, positions):
    seen, labels = {}, []
    for s in panel['Short']:
        seen[s] = seen.get(s, 0) + 1
        labels.append(s if seen[s] == 1 else f'{s} ({seen[s]})')
    panel = panel.assign(Label=labels).iloc[::-1].reset_index(drop=True)
    fig.add_trace(go.Bar(
        x=panel['Importance'], y=panel['Label'], orientation='h',
        marker_color=[CATEGORY_COLORS.get(cat, '#888') for cat in panel['Category']],
        hovertext=panel['Feature'], hoverinfo='text+x', showlegend=False,
    ), row=r, col=c)
    fig.update_yaxes(categoryorder='array', categoryarray=panel['Label'].tolist(),
                     row=r, col=c, automargin=True, tickfont=dict(size=10))
    fig.update_xaxes(title_text='Importance', row=r, col=c)

for cat, color in CATEGORY_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], name=cat,
                         marker_color=color, showlegend=True, hoverinfo='skip'),
                  row=1, col=1)
fig.update_layout(
    title='Top-15 features per method (L5B15) — Pega NB vs surrogate explainers',
    height=1400, width=1200, template='plotly_white',
    legend=dict(title='Category', orientation='h', y=-0.05),
    margin=dict(l=10, r=10, t=80, b=80),
    barmode='group',
)
fig.show()


## 5. Talking points for the Pega DS team

- **Pega NB importance is the *ground truth* feature importance for this model** — it's exactly what Prediction Studio displays. We can now position SHAP/LIME/DT and the vendor XGBoost as *approximations* of it, and quantify their fidelity.
- **If Spearman ρ(Pega NB, SHAP) is high**, that justifies using TreeSHAP as a more flexible explainer (e.g. for non-NB models, or for per-instance explanations) without losing alignment with Pega's official ranking.
- **If ρ(Pega NB, vendor XGBoost) is also high**, it argues that the vendor's HDS-style XGBoost can be used as a feature-discovery tool (find new candidate predictors) while still reflecting the same importance ordering Pega itself would assign.
- **Where the methods *disagree***, that is exactly the δ_e (explainer sensitivity) signal from RQ3: even on the same model, different explanation lenses produce different rankings. Quantifying that gap is what allows you to set a noise floor before treating a feature-ranking change as real.